# SenGuard-MSC kernel F: virtual passage
Needs kernel E attached. Reuses the cohort loaders from notebook 06 unchanged.

In [ ]:
# Restores the frozen outputs of kernel A at their original absolute paths, so hash checks stay meaningful.
import shutil
from pathlib import Path
hits = sorted(Path('/kaggle/input').rglob('frozen_model.json'))
if not hits:
    raise FileNotFoundError('Kernel A output is not attached: no frozen_model.json under /kaggle/input.')
src = hits[0].parents[1]
dst = Path('/kaggle/working/senguard_msc')
if not dst.exists():
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.git', '.venv_check'))
print('Restored', src, '->', dst, '|', sorted(p.name for p in dst.iterdir()))

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], capture_output=True, text=True).returncode)

SenGuard-MSC · Notebook 06 — Protocol amendment: more MSC cohorts, head-to-head, robustness, drift clock

Written after G3 was seen, so this is an amendment and is labelled as one. Cohorts below were chosen from metadata before any of them was scored. The amendment is written to disk and hashed before any expression file is read.

In [ ]:
import gc, gzip, io, json, os, re, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests
ROOT_GUESS = Path('/kaggle/working/senguard_msc') if Path('/kaggle/working/senguard_msc').exists() else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, EXT, DEV, ASC, BM
from scipy import stats
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
RAWX = ROOT / 'data/raw_extended'
RAWX.mkdir(parents=True, exist_ok=True)
FREEZE = json.loads((RESULTS / 'frozen_model.json').read_text())
for name, art in FREEZE['artifacts'].items():
    if sg.sha256(art['path']) != art['sha256']:
        raise RuntimeError(f'{name} changed after the freeze')
print('Frozen artifacts verified. Model weights and published SenCat weights are unchanged.')

AMENDMENT = """# Analysis protocol amendment 1

Fixed on 2026-11-01, after G3 results on GSE309666 and GSE178514 were seen, and before any cohort below was scored.
Inclusion came from GEO metadata only. Published and refit weights stay frozen.

## New cohorts
| Cohort | Tissue | Design | Use |
|---|---|---|---|
| GSE137186 | BM | 4 donors, P1, P5, P10, P15, 2 resequenced samples | Pooled; test-retest; authors report retained immunoregulatory potency |
| GSE275980 | MSC, tissue unstated in the abstract | 3 donors, P2, P6, P10 | Pooled |
| GSE125632 | BM | 5 donors, P2 vs P11 labelled senescent by authors | Pooled; labelled check |
| GSE194298 | BM | 4 donors, p1 vs p4/5, FBS and hPL | Pooled per donor; medium effect |
| GSE119987 | UC | 4 donors to replicative senescence, array, doubling level known | Pooled; early change |
| GSE56362 | BM | Donors at P3, P5, P7 with technical replicates, array | Pooled; test-retest |
| GSE233634 | UC | P3 to P12 in triplicate, no donor stated, counts as one unit | Pooled as one lineage |
| GSE269586 | WJ | GMP production, master bank to P2 or P3 | Not pooled: short-expansion comparison |
| GSE48662 | BM | Irradiated vs control; GVHD responders G1 (5) vs non-responders G2 (3), array | Labelled check; clinical-response question |

## Primary question
Across donors in new pooled cohorts, does the published SenCat score rise with passage? Test: two-sided exact binomial on donors, plus a cohort-level sign-flip test. Log1p weighted score on RNA-seq; rank-weighted form on arrays. Sensitivities: rank form everywhere, weights refit without BMMSC and preadipocytes, excluding GSE56362 (gene-normalized values), and excluding GSE233634.

## Head-to-head
Published SenCat, hUSI, SenMayo, the 7-gene signature of Yang 2023, the 40 AD-MSC biomarkers, Hallmark E2F and G2M (inverted), and CDKN2A, CDKN1A, IL6, GDF15, LMNB1 (inverted) transcripts, all in rank form. Reported: pooled donors rising, median within-donor rho, cohorts with a positive mean. Results on a score's own training cohort are flagged and left out of its pooled count. No single winner is declared unless its lead is clear of the uncertainty.

## Clinical-response question
One directional hypothesis: published SenCat rank score is higher in GVHD non-responders (G2) than responders (G1). Exact permutation, smallest one-sided p 1/56. The outcome will be reported either way, and it would be hypothesis-generating at any p.

## Labelled MSC checks
AUROC is allowed only where the authors define the classes: GSE48662 control vs irradiated, GSE125632 P2 vs P11 senescent. Within-cohort only, with no transported threshold.

## Drift clock (trained model)
Ridge regression of within-donor log2(passage + 1) on within-sample gene ranks, with leave-one-cohort-out validation and all selection inside training cohorts. It estimates passage-like drift and is not a senescence or potency measure. Negative control: passages shuffled within donors in training. It enters the demo only if held-out median within-donor rho is at least 0.8, at least 90% of held-out donors rise, and the shuffled control's median rho stays below 0.3.

## Robustness
Binomial thinning of counts, random gene dropout, technical test-retest, leave-one-sample-out normalization, medium effect, and absolute-score comparability across cohorts. These set guard thresholds in the tool from measurement instead of judgement.
"""
amend_path = REPORTS / 'ANALYSIS_AMENDMENT_1.md'
amend_path.write_text(AMENDMENT, encoding='utf-8')
print('Amendment sha256', sg.sha256(amend_path))

## Download and parse each cohort
Each cohort loads inside its own try block. A failure is recorded and doesn't stop the others.

In [ ]:
UA = {'User-Agent': 'SenGuard-MSC-audit/1.0'}
GEO = 'https://ftp.ncbi.nlm.nih.gov/geo'
CGI = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi'

def fetch(url, name=None):
    dest = RAWX / (name or url.rsplit('/', 1)[-1])
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    for attempt in range(5):
        try:
            if attempt < 3:
                with requests.get(url, headers=UA, stream=True, timeout=300) as r:
                    r.raise_for_status()
                    with open(dest, 'wb') as f:
                        for c in r.iter_content(1 << 20):
                            f.write(c)
            else:
                subprocess.run(['curl', '-fsSL', '--http1.1', '-A', UA['User-Agent'], '-o', str(dest), url], check=True, timeout=600)
            if dest.stat().st_size > 0:
                if dest.suffix == '.gz':
                    with gzip.open(dest, 'rb') as f:
                        while f.read(1 << 20):
                            pass
                return dest
        except Exception as e:
            print('  retry', dest.name, type(e).__name__, str(e)[:90])
        time.sleep(4 * (attempt + 1))
    raise RuntimeError(f'Could not download {url}')

def sdir(acc):
    return f'{GEO}/series/{acc[:-3]}nnn/{acc}'

def parse_matrix(path):
    meta, chars = {}, []
    rows = []
    in_table = False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!series_matrix_table_begin'):
                in_table = True
            elif line.startswith('!series_matrix_table_end'):
                break
            elif in_table:
                rows.append(line.rstrip('\n').replace('"', '').split('\t'))
            elif line.startswith('!Sample_'):
                parts = line.rstrip('\n').replace('"', '').split('\t')
                if parts[0] == '!Sample_characteristics_ch1':
                    chars.append(parts[1:])
                else:
                    meta[parts[0][8:]] = parts[1:]
    expr = pd.DataFrame(rows[1:], columns=rows[0]).set_index('ID_REF').apply(pd.to_numeric, errors='coerce').T
    m = pd.DataFrame({'sample_id': meta['geo_accession'], 'title': meta['title']})
    for col in chars:
        key = col[0].split(':')[0].strip().lower()
        m[key] = [c.split(':', 1)[1].strip() if ':' in c else c for c in col]
    return expr, m.set_index('sample_id')

def probe_symbols(gpl):
    annot = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/annot/{gpl}.annot.gz'
    soft = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/soft/{gpl}_family.soft.gz'
    try:
        path = fetch(annot, f'{gpl}.annot.gz')
    except Exception:
        path = fetch(soft, f'{gpl}_family.soft.gz')
    rows, header, on = [], None, False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!platform_table_begin'):
                on = True
                continue
            if line.startswith('!platform_table_end'):
                break
            if on:
                parts = line.rstrip('\n').split('\t')
                if header is None:
                    header = parts
                elif len(parts) == len(header):
                    rows.append(parts)
    t = pd.DataFrame(rows, columns=header)
    col = next((c for c in t.columns if c.lower() in ('gene symbol', 'gene_symbol', 'genesymbol', 'symbol', 'gene_name')), None)
    if col is None and 'gene_assignment' in t.columns:
        t['sym'] = t.gene_assignment.str.split('//').str[1].str.strip()
        col = 'sym'
    if col is None:
        raise RuntimeError(f'{gpl}: no symbol column in {list(t.columns)[:15]}')
    s = t.set_index('ID')[col].astype(str).str.strip()
    s = s[(s != '') & (s != 'nan') & (s != '---') & ~s.str.contains('///')]
    return s

ens2sym_tbl = pd.read_csv(PROC / f'{ASC}_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
sym2ens_tbl = pd.Series(ens2sym_tbl.index, index=ens2sym_tbl.values)
sym2ens_tbl = sym2ens_tbl[~sym2ens_tbl.index.duplicated(keep=False)]

def to_sym(expr):
    """Samples x genes to samples x symbols; Ensembl ids are mapped through annotation, keeping the highest-mean gene per symbol."""
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        c = cols.str.replace(r'\.\d+$', '', regex=True)
        e = expr.copy(); e.columns = c
        return sg.to_symbols(e.loc[:, ~e.columns.duplicated()], pd.DataFrame({'symbol': ens2sym_tbl.reindex(c.unique())}, index=c.unique()))
    e = expr.loc[:, ~pd.Index(cols).duplicated()]
    return e

def to_ens(expr):
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        e = expr.copy(); e.columns = cols.str.replace(r'\.\d+$', '', regex=True)
        return e.loc[:, ~e.columns.duplicated()]
    m = sym2ens_tbl.reindex(cols)
    e = expr.loc[:, m.notna().to_numpy()].copy()
    e.columns = m.dropna().to_numpy()
    return e.loc[:, ~e.columns.duplicated()]

COHORTS, LOAD_LOG = {}, []
def add(key, expr, meta, kind, tissue, pooled, role, note=''):
    """kind: counts, normcounts, fpkm, log_array."""
    meta = meta.loc[expr.index].copy()
    if not {'subject', 'passage', 'unit'} <= set(meta.columns):
        raise ValueError(f'{key}: meta needs subject, passage, unit; has {list(meta.columns)}')
    meta['passage'] = meta.passage.astype(float)
    COHORTS[key] = dict(expr=expr, meta=meta, kind=kind, tissue=tissue, pooled=pooled, role=role, note=note)
    LOAD_LOG.append({'cohort': key, 'status': 'loaded', 'samples': len(expr), 'genes': expr.shape[1], 'subjects': meta.subject.nunique(), 'units': meta.unit.nunique(), 'kind': kind, 'tissue': tissue, 'pooled': pooled, 'note': note})
    print(f'{key}: {expr.shape} subjects {meta.subject.nunique()} units {meta.unit.nunique()} passages {sorted(meta.passage.unique())[:10]} kind {kind}')

def attempt(key, fn):
    try:
        fn()
    except Exception as e:
        LOAD_LOG.append({'cohort': key, 'status': f'failed: {type(e).__name__}: {str(e)[:220]}'})
        print(key, 'FAILED', type(e).__name__, str(e)[:300])

def geo_samples(acc):
    t = requests.get(CGI, params={'acc': acc, 'targ': 'gsm', 'form': 'text', 'view': 'full'}, headers=UA, timeout=180).text
    rows = []
    for b in re.split(r'\^SAMPLE = ', t)[1:]:
        ch = {c.split(':')[0].strip().lower(): c.split(':', 1)[1].strip() for c in re.findall(r'!Sample_characteristics_ch1 = (.*)', b) if ':' in c}
        rows.append({'gsm': b.split('\n')[0].strip(), 'title': re.findall(r'!Sample_title = (.*)', b)[0].strip(), 'description': ' | '.join(x.strip() for x in re.findall(r'!Sample_description = (.*)', b)), **ch})
    time.sleep(0.5)
    return pd.DataFrame(rows)

# Already audited cohorts
def _asc():
    m = sg.load_manifest(); a = m[(m.study_accession == ASC)]
    e = pd.read_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    meta = pd.DataFrame({'subject': a.donor.astype(str), 'unit': a.donor.astype(str), 'passage': a.passage}, index=a.index)
    add(ASC, e, meta, 'normalized', 'AD', False, 'seen')
attempt(ASC, _asc)
def _bm():
    m = sg.load_manifest(); a = m[(m.study_accession == BM) & m.use_in_mrna_analysis]
    e = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    ids = a.candidate_biological_id.astype(int).astype(str)
    add(BM, e, pd.DataFrame({'subject': ids, 'unit': ids, 'passage': a.passage}, index=a.index), 'fpkm', 'BM', False, 'seen', 'Donors inferred')
attempt(BM, _bm)

def _137():
    p = fetch(f'{sdir("GSE137186")}/suppl/GSE137186_raw.counts.txt.gz')
    t = pd.read_csv(p, sep='\t', header=None, dtype=str, low_memory=False)
    head = t.iloc[:4]
    body = t.iloc[4:]
    body = body[~body[1].astype(str).str.startswith('__')]
    expr = body.set_index(1).iloc[:, 1:].apply(pd.to_numeric, errors='coerce').T
    names = [f'S{i}' for i in head.iloc[1, 2:]]
    expr.index = names
    meta = pd.DataFrame({'subject': head.iloc[2, 2:].to_numpy(), 'unit': head.iloc[2, 2:].to_numpy(), 'passage': head.iloc[3, 2:].astype(float).to_numpy()}, index=names)
    expr = expr.loc[:, ~expr.columns.duplicated()]
    add('GSE137186', expr, meta, 'counts', 'BM', True, 'new', 'Authors report retained immunoregulatory potency')
attempt('GSE137186', _137)

def _275():
    p = fetch(f'{sdir("GSE275980")}/suppl/GSE275980_raw_count.xlsx')
    t = pd.read_excel(p)
    t = t.drop_duplicates(t.columns[0]).set_index(t.columns[0])
    t.columns = [c.replace('P2C2', 'P2-C2') for c in t.columns]
    expr = t.apply(pd.to_numeric, errors='coerce').fillna(0).T
    parts = pd.Series(expr.index).str.extract(r'P(\d+)-C(\d)')
    meta = pd.DataFrame({'subject': ('C' + parts[1]).to_numpy(), 'unit': ('C' + parts[1]).to_numpy(), 'passage': parts[0].astype(float).to_numpy()}, index=expr.index)
    add('GSE275980', expr, meta, 'counts', 'MSC', True, 'new', 'Gene symbols only; C1-C3 are the three donors in the series title')
attempt('GSE275980', _275)

def _125():
    p = fetch(f'{sdir("GSE125632")}/suppl/GSE125632_FPKM_iMSC_manuscript.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False, on_bad_lines='skip', dtype=str)
    t = t[t.EnsemblID.astype(str).str.startswith('ENSG')].drop_duplicates('EnsemblID').set_index('EnsemblID')
    cols = [f'Sample_{i}' for i in range(1, 21)]
    expr = t[cols].apply(pd.to_numeric, errors='coerce').fillna(0).T
    g = geo_samples('GSE125632')
    meta = pd.DataFrame({'title': g.title.to_numpy(), 'subject': g.donor.to_numpy(), 'passage': g.passage.str.replace('P', '').astype(float).to_numpy()}, index=cols)
    meta['arm'] = np.where(meta.title.str.startswith('iMSC'), 'iMSC', 'MSC')
    meta['gender'] = g.gender.to_numpy()
    y = t.reindex(['ENSG00000129824'])[cols].apply(pd.to_numeric, errors='coerce').iloc[0]
    agree = float(((y > 1).to_numpy() == (meta.gender == 'Male').to_numpy()).mean())
    print('  GSE125632 column order check: RPS4Y1 agrees with GEO sex in', f'{agree:.0%}', 'of samples')
    keep = meta.arm == 'MSC'
    m = meta[keep].assign(unit=lambda d: d.subject)
    add('GSE125632', expr.loc[keep], m, 'fpkm', 'BM', True, 'new', f'Column order to GSM inferred; sex check agreement {agree:.0%}')
    im = meta[~keep].assign(unit=lambda d: d.subject)
    add('GSE125632_iMSC', expr.loc[~keep], im, 'fpkm', 'iPSC-derived', False, 'new', 'Reported separately')
attempt('GSE125632', _125)

def _194():
    p = fetch(f'{sdir("GSE194298")}/suppl/GSE194298_Norm_counts_RNAseq_hPL_vs_FBS.csv.gz')
    t = pd.read_csv(p, index_col=0)
    t.columns = [re.search(r'X(\d+)', c).group(1) for c in t.columns]
    g = geo_samples('GSE194298')
    g['num'] = g.title.str.extract(r'_(\d+)$')
    g = g.set_index('num').loc[t.columns]
    expr = t.T
    medium = g.treatment.str.strip().to_numpy()
    meta = pd.DataFrame({'donor': g.donor.to_numpy(), 'medium': medium, 'passage': g.passage.map({'p1': 1.0, 'p4/p5': 4.5}).to_numpy()}, index=t.columns)
    meta['subject'] = meta.donor + '_' + meta.medium
    meta['unit'] = meta.donor
    add('GSE194298', expr, meta, 'normcounts', 'BM', True, 'new', 'Medium arms averaged per donor; mBG arm has no p1')
attempt('GSE194298', _194)

def _233():
    p = fetch(f'{sdir("GSE233634")}/suppl/GSE233634_gene_FPKM.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('geneID')
    plane = [c for c in t.columns if re.fullmatch(r'P\d+_\d', c)]
    bio = [c for c in t.columns if re.fullmatch(r'BioR\d_\d', c)]
    expr = t[plane].apply(pd.to_numeric, errors='coerce').T
    meta = pd.DataFrame({'passage': pd.Series(plane).str.extract(r'P(\d+)_')[0].astype(float).to_numpy(), 'subject': 'lineage', 'unit': 'lineage', 'rep': pd.Series(plane).str.extract(r'_(\d)')[0].to_numpy()}, index=plane)
    add('GSE233634', expr, meta, 'fpkm', 'UC', True, 'new', 'No donor stated; one unit')
    COHORTS['GSE233634']['bioreactor'] = t[bio].apply(pd.to_numeric, errors='coerce').T
attempt('GSE233634', _233)

def _269():
    p = fetch(f'{sdir("GSE269586")}/suppl/GSE269586_all_counts_merged.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('id')
    g = geo_samples('GSE269586')
    g = g[g.title.str.contains('RNAseq')].copy()
    g['col'] = 'raw_' + g.description.str.split(' \\| ').str[0]
    missing = [c for c in g.col if c not in t.columns]
    if missing:
        raise RuntimeError(f'columns missing: {missing[:4]}')
    expr = t[g.col.tolist()].apply(pd.to_numeric, errors='coerce').fillna(0).T
    expr.index = g.gsm.to_numpy()
    donor = g.donor.str.replace('donor ', '').to_numpy()
    meta = pd.DataFrame({'subject': donor, 'unit': donor, 'passage': g.passage.astype(float).to_numpy()}, index=g.gsm.to_numpy())
    add('GSE269586', expr, meta, 'counts', 'WJ', False, 'new', 'GMP production; short expansion; not pooled')
attempt('GSE269586', _269)

def _119():
    p = fetch(f'{sdir("GSE119987")}/matrix/GSE119987_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL571')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    meta = pd.DataFrame({'subject': m['donor population name'], 'unit': m['donor population name'].str.replace(r'\.\d+$', '', regex=True), 'passage': m['passage number'].astype(float), 'pdl': pd.to_numeric(m.get('population doubling level'), errors='coerce'), 'batch': m.get('overall batch')}, index=m.index)
    add('GSE119987', e, meta, 'log_array', 'UC', True, 'new', 'Cultured to replicative senescence')
attempt('GSE119987', _119)

def _563():
    p = fetch(f'{sdir("GSE56362")}/matrix/GSE56362_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL18503')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    has = m['passage'].astype(str).str.contains(r'\d')
    meta = pd.DataFrame({'subject': m['donor id'], 'unit': m['donor id'], 'passage': m['passage'].astype(str).str.extract(r'(\d+)')[0].astype(float)}, index=m.index)[has]
    e = e.loc[meta.index]
    add('GSE56362', e, meta, 'log_array', 'BM', True, 'new', 'Values look gene-normalized; technical triplicates averaged')
attempt('GSE56362', _563)

def _486():
    p = fetch(f'{sdir("GSE48662")}/matrix/GSE48662-GPL13497_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL13497')
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    m['group'] = m.title.str.extract(r'MSC[_-](WT|SEN|G1|G2)')[0]
    m = m[m.group.notna()]
    meta = m[['title', 'group']].copy()
    meta['subject'] = m.title.str.extract(r'-(\d+)')[0].to_numpy()
    meta['unit'] = meta.subject
    meta['passage'] = np.nan
    COHORTS['GSE48662'] = dict(expr=e.loc[m.index], meta=meta, kind='log_array', tissue='BM', pooled=False, role='new', note='Control vs irradiated, and GVHD responder groups')
    LOAD_LOG.append({'cohort': 'GSE48662', 'status': 'loaded', 'samples': len(m), 'genes': e.shape[1], 'kind': 'log_array', 'tissue': 'BM', 'pooled': False, 'note': str(m.group.value_counts().to_dict())})
    print('GSE48662', e.loc[m.index].shape, m.group.value_counts().to_dict())
attempt('GSE48662', _486)
pd.DataFrame(LOAD_LOG).to_csv(RESULTS / 'ext_cohort_load_log.csv', index=False)
print(pd.DataFrame(LOAD_LOG)[['cohort', 'status', 'samples', 'genes', 'subjects', 'units', 'kind']].to_string(index=False))

## Signatures and scoring
Weights and gene lists are fixed. Rank forms are computed for all cohorts, so arrays and RNA-seq can be compared. For RNA-seq the published log1p weighted form is also computed, after the median-of-ratios step where counts allow it.

In [ ]:
pub_tbl = sg.load_sencat_markers()
PUB = pub_tbl.coef
pub_sym = pub_tbl.dropna(subset=['gene']).drop_duplicates('gene').set_index('gene').coef
REFIT = pd.read_csv(RESULTS / 'frozen_weights_refit_no_BMMSC_prAD.csv').set_index('gene_id').coef
refit_sym = REFIT.copy(); refit_sym.index = ens2sym_tbl.reindex(REFIT.index).to_numpy(); refit_sym = refit_sym[refit_sym.index.notna()]; refit_sym = refit_sym[~refit_sym.index.duplicated()]
HUSI = sg.load_husi_weights()
hall, extra = sg.load_gene_sets()
YANG7 = ['PRUNE2', 'DIO2', 'CPA4', 'PRKAA2', 'DMD', 'DDAH1', 'GATA6']
UP40 = 'CACNB4 CRYAB DAW1 EFEMP1 EPAS1 GALNT5 H1-2 KRT34 MELTF MEST MFGE8 MGARP MOXD1 SERPINB2 SERPINE2 SLC43A2 SRGN SVEP1 SYNPO2 TMEM178B'.split()
DOWN40 = 'AURKB BUB1 BUB1B CDC20 DEPDC1 KIF14 KIF18B KIF2C KIFC1 KNL1 LMNB1 NCAPH NUF2 PIMREG PKMYT1 SKA3 TNC TROAP TTK UBE2C'.split()

def rank_weighted(sym, w):
    shared = [g for g in w.index if g in sym.columns]
    wr = stats.rankdata(w.loc[shared].to_numpy())
    r = np.apply_along_axis(stats.rankdata, 1, sym[shared].fillna(0).to_numpy(dtype=float))
    wc, rc = wr - wr.mean(), r - r.mean(axis=1, keepdims=True)
    return pd.Series((rc @ wc) / (np.linalg.norm(rc, axis=1) * np.linalg.norm(wc)), index=sym.index), len(shared)

def primary_input(c):
    e = to_ens(c['expr'])
    if c['kind'] == 'counts':
        n = sg.mor_normalize(sg.author_gene_filter(e))[0]
    elif c['kind'] == 'fpkm':
        n = sg.scale_to_million(e)
    else:
        n = e
    return n

SCORES, COVER = {}, []
for key, c in COHORTS.items():
    sym = to_sym(c['expr'])
    sym = sym.fillna(sym.median()).fillna(0) if c['kind'] == 'log_array' else sym.fillna(0)
    c['sym'] = sym
    out = {}
    if c['kind'] != 'log_array':
        n = primary_input(c)
        c['norm'] = n
        out['SenCat published'], u1 = sg.weighted_log1p_score(n, PUB)
        out['SenCat refit, lineage excluded'], _ = sg.weighted_log1p_score(n, REFIT)
    else:
        out['SenCat published'], u1 = rank_weighted(sym, pub_sym)
        out['SenCat refit, lineage excluded'], _ = rank_weighted(sym, refit_sym)
    out['SenCat published, rank form'], u2 = rank_weighted(sym, pub_sym)
    out['hUSI'], u3 = sg.husi_raw(sym, HUSI)
    ranks = sym.rank(axis=1, pct=True)
    for nm, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Yang 2023 7 genes', YANG7, 1), ('Hallmark E2F (inverted)', hall['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', hall['HALLMARK_G2M_CHECKPOINT'], -1)]:
        have = [g for g in genes if g in ranks.columns]
        out[nm] = sign * ranks[have].mean(axis=1)
        COVER.append({'cohort': key, 'score': nm, 'genes_present': len(have), 'genes_total': len(genes)})
    hu, hd = [g for g in UP40 if g in ranks.columns], [g for g in DOWN40 if g in ranks.columns]
    out['AD-MSC 40 biomarkers'] = ranks[hu].mean(axis=1) - ranks[hd].mean(axis=1)
    for g, sign in [('CDKN2A', 1), ('CDKN1A', 1), ('IL6', 1), ('GDF15', 1), ('LMNB1', -1)]:
        if g in ranks.columns:
            out[g + (' (inverted)' if sign < 0 else '')] = sign * ranks[g]
    COVER.append({'cohort': key, 'score': 'SenCat published', 'genes_present': int(u1), 'genes_total': len(PUB)})
    COVER.append({'cohort': key, 'score': 'hUSI', 'genes_present': int(u3), 'genes_total': len(HUSI)})
    SCORES[key] = pd.DataFrame(out)
pd.DataFrame(COVER).to_csv(RESULTS / 'ext_signature_coverage.csv', index=False)
cov = pd.DataFrame(COVER).assign(pct=lambda d: (100 * d.genes_present / d.genes_total).round(1)).pivot(index='cohort', columns='score', values='pct')
print(cov.to_string())

husi_meta = pd.read_csv(EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')
all_text = husi_meta.astype(str).agg(' '.join, axis=1)
husi_overlap = {k.split('_')[0]: int(all_text.str.contains(k.split('_')[0]).sum()) for k in COHORTS}
print('hUSI training rows by cohort accession:', husi_overlap)
CIRCULAR = {'hUSI': {k for k, v in husi_overlap.items() if v > 0}, 'AD-MSC 40 biomarkers': {ASC}, 'Yang 2023 7 genes': {BM}}
json.dump({'husi_rows_by_accession': husi_overlap, 'circular': {k: sorted(v) for k, v in CIRCULAR.items()}}, open(RESULTS / 'ext_circularity.json', 'w'), indent=1)

long = []
for key, s in SCORES.items():
    m = COHORTS[key]['meta']
    x = s.join(m[['subject', 'unit', 'passage']]).rename_axis('sample').reset_index().melt(['sample', 'subject', 'unit', 'passage'], var_name='score_name', value_name='score')
    x.insert(0, 'cohort', key)
    long.append(x)
pd.concat(long).to_csv(RESULTS / 'ext_scores_long.csv', index=False)

SenGuard-MSC · Notebook 07 — Virtual passage: predicting a culture's later expression

Can an early sample predict the same culture's expression profile at a later passage? This is the predictive part of the twin. The rules below are written and hashed before any prediction is evaluated.

In [ ]:
AMENDMENT2 = """# Analysis protocol amendment 2: virtual passage

Fixed on 2026-11-01, before any prediction below was evaluated. It follows a failed per-donor forecast (deliverables/tables/forecast_backtest.json), so the idea of learning across cohorts was chosen after seeing that failure.

## Question
From one early sample of a culture, predict its within-sample gene ranks at a later passage.

## Models
- Carry-forward: no change. This is the baseline to beat.
- Population drift: a per-gene drift rate learned from other cohorts, multiplied by the change in log2(passage + 1).
- Personal drift: population drift scaled by how fast this donor moved between its first two passages.

## Validation
Leave one cohort out. Cohorts: GSE309666, GSE178514, GSE137186, GSE275980, GSE194298, GSE233634, GSE119987, GSE56362. GSE125632 is excluded because it failed the identity check. Unit: donor (medium arms averaged). Earliest available passage predicts last available passage.

## Gate for the interface
Population drift enters the interface only if all of these hold:
1. Gene-level RMSE beats carry-forward in at least 70% of held-out donors.
2. It wins in at least 6 of the 8 cohorts, by median error.
3. Predicted SenCat rank score has lower absolute error than carry-forward in at least 70% of held-out donors.
4. Median correlation between predicted and actual change across genes is at least 0.3.
5. Negative control: drift learned from passages shuffled within donors beats carry-forward in at most 60% of donors on RMSE, on average.

Personal drift is used only if it beats population drift in at least 60% of donors with 3 or more passages, on the later step.

## What this does not claim
It predicts transcript ranks. It does not predict potency, growth arrest, safety or cell behaviour. Remaining expansion capacity is examined only descriptively, because too few lineages have a growth endpoint.
"""

In [ ]:
HELD = ['GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362']
(REPORTS / 'ANALYSIS_AMENDMENT_2.md').write_text(AMENDMENT2, encoding='utf-8')
print('Amendment 2 sha256', sg.sha256(REPORTS / 'ANALYSIS_AMENDMENT_2.md'))
missing = [k for k in HELD if k not in COHORTS]
print('Missing cohorts:', missing)
HELD = [k for k in HELD if k in COHORTS]
genes = None
for k in HELD:
    cols = set(COHORTS[k]['sym'].columns)
    genes = cols if genes is None else genes & cols
genes = sorted(genes)
print('Shared genes', len(genes))
PROFILES = {}
for k in HELD:
    c = COHORTS[k]
    R = c['sym'][genes].rank(axis=1, pct=True)
    d = R.join(c['meta'][['subject', 'unit', 'passage']]).groupby(['subject', 'unit', 'passage'], as_index=False).mean(numeric_only=True)
    subs = []
    for (s, u), g in d.groupby(['subject', 'unit']):
        g = g.sort_values('passage')
        if g.passage.nunique() >= 2:
            subs.append(dict(subject=s, unit=u, passage=g.passage.to_numpy(float), X=g[genes].to_numpy(np.float32)))
    PROFILES[k] = subs
    print(k, 'subjects', len(subs))
tlog = lambda p: np.log2(np.asarray(p, float) + 1)
weights = pub_sym.reindex(genes).dropna()
w_idx = [genes.index(g) for g in weights.index]
w_rank = stats.rankdata(weights.to_numpy())
print('SenCat genes among shared', len(w_idx))
def rank_score(x):
    r = stats.rankdata(x[w_idx])
    return float(np.corrcoef(r, w_rank)[0, 1])

def learn_beta(cohorts, rng=None):
    per = []
    for k in cohorts:
        bs = []
        for s in PROFILES[k]:
            t = tlog(s['passage'])
            if rng is not None:
                t = rng.permutation(t)
            tc = t - t.mean()
            denom = (tc ** 2).sum()
            if denom > 0:
                bs.append((tc[None, :] @ (s['X'] - s['X'].mean(axis=0))).ravel() / denom)
        if bs:
            per.append(np.mean(bs, axis=0))
    return np.mean(per, axis=0)

rows, personal = [], []
betas = {}
for k in HELD:
    train = [c for c in HELD if c != k]
    beta = learn_beta(train)
    betas[k] = beta
    for s in PROFILES[k]:
        x0, xL = s['X'][0], s['X'][-1]
        dt = tlog(s['passage'][-1]) - tlog(s['passage'][0])
        y = xL - x0
        pred = beta * dt
        rmse_m, rmse_0 = float(np.sqrt(np.mean((y - pred) ** 2))), float(np.sqrt(np.mean(y ** 2)))
        sc_true, sc_0, sc_m = rank_score(xL), rank_score(x0), rank_score(x0 + pred)
        rows.append({'cohort': k, 'subject': s['subject'], 'unit': s['unit'], 'from': s['passage'][0], 'to': s['passage'][-1], 'dt': dt, 'rmse_model': rmse_m, 'rmse_carry': rmse_0, 'corr_change': float(np.corrcoef(pred, y)[0, 1]),
                     'score_true': sc_true, 'score_carry': sc_0, 'score_model': sc_m, 'err_score_model': abs(sc_m - sc_true), 'err_score_carry': abs(sc_0 - sc_true)})
        if len(s['passage']) >= 3:
            xm = s['X'][1]
            t0, tm, tL = tlog(s['passage'][0]), tlog(s['passage'][1]), tlog(s['passage'][-1])
            gamma = float(np.clip(((xm - x0) @ beta) / ((beta @ beta) * (tm - t0)), 0, 5))
            late = float(((xL - xm) @ beta) / ((beta @ beta) * (tL - tm)))
            y2 = xL - xm
            personal.append({'cohort': k, 'subject': s['subject'], 'unit': s['unit'], 'gamma_early': gamma, 'gamma_late': late, 'rmse_carry': float(np.sqrt(np.mean(y2 ** 2))), 'rmse_population': float(np.sqrt(np.mean((y2 - beta * (tL - tm)) ** 2))), 'rmse_personal': float(np.sqrt(np.mean((y2 - gamma * beta * (tL - tm)) ** 2)))})
res = pd.DataFrame(rows)
units = res.groupby(['cohort', 'unit'], as_index=False).mean(numeric_only=True)
units['model_better'] = units.rmse_model < units.rmse_carry
units['score_better'] = units.err_score_model < units.err_score_carry
units.to_csv(RESULTS / 'virtual_passage_units.csv', index=False)
print(units[['cohort', 'unit', 'from', 'to', 'rmse_carry', 'rmse_model', 'corr_change', 'err_score_carry', 'err_score_model', 'model_better', 'score_better']].round(3).to_string(index=False))
by = units.groupby('cohort').agg(donors=('unit', 'size'), rmse_carry=('rmse_carry', 'median'), rmse_model=('rmse_model', 'median'), corr=('corr_change', 'median'), better=('model_better', 'sum'), score_better=('score_better', 'sum'), err_carry=('err_score_carry', 'median'), err_model=('err_score_model', 'median'))
print(by.round(3).to_string())

rng = np.random.default_rng(sg.SEED)
neg = []
for rep in range(20):
    wins = []
    for k in HELD:
        beta = learn_beta([c for c in HELD if c != k], rng=rng)
        for s in PROFILES[k]:
            y = s['X'][-1] - s['X'][0]
            dt = tlog(s['passage'][-1]) - tlog(s['passage'][0])
            wins.append(np.mean((y - beta * dt) ** 2) < np.mean(y ** 2))
    neg.append(float(np.mean(wins)))
pers = pd.DataFrame(personal)
pers_u = pers.groupby(['cohort', 'unit'], as_index=False).mean(numeric_only=True)
pers_u.to_csv(RESULTS / 'virtual_passage_personal.csv', index=False)
print(pers_u.round(4).to_string(index=False))
persist = float(stats.spearmanr(pers_u.gamma_early, pers_u.gamma_late).statistic) if len(pers_u) > 2 else float('nan')

gate = {
    'donors': len(units),
    'share_rmse_better': float(units.model_better.mean()),
    'cohorts_better': int((by.rmse_model < by.rmse_carry).sum()), 'cohorts': len(by),
    'share_score_better': float(units.score_better.mean()),
    'median_corr_change': float(units.corr_change.median()),
    'negative_control_share': float(np.mean(neg)),
    'median_rmse_reduction': float(1 - (units.rmse_model / units.rmse_carry).median()),
    'median_err_score_model': float(units.err_score_model.median()), 'median_err_score_carry': float(units.err_score_carry.median()),
    'personal_share_better': float((pers_u.rmse_personal < pers_u.rmse_population).mean()), 'personal_donors': len(pers_u), 'rate_persistence_spearman': persist,
    'population_better_than_carry_later_step': float((pers_u.rmse_population < pers_u.rmse_carry).mean()),
}
gate['passes'] = bool(gate['share_rmse_better'] >= 0.7 and gate['cohorts_better'] >= 6 and gate['share_score_better'] >= 0.7 and gate['median_corr_change'] >= 0.3 and gate['negative_control_share'] <= 0.6)
gate['personal_passes'] = bool(gate['personal_share_better'] >= 0.6)
print(json.dumps(gate, indent=1))

# Error for the interface, by how far ahead the prediction reaches
units['dt_bin'] = pd.cut(units.dt, [0, 1, 2, 10], labels=['under 1', '1 to 2', 'over 2'])
err = units.groupby('dt_bin', observed=True).agg(n=('unit', 'size'), rmse=('rmse_model', 'median'), score_err=('err_score_model', 'median'), score_err_p90=('err_score_model', lambda v: float(np.percentile(v, 90))))
print(err.round(4).to_string())
final_beta = learn_beta(HELD)
top = pd.Series(final_beta, index=genes).sort_values()
print('Falling with passage:', top.head(25).index.tolist())
print('Rising with passage:', top.tail(25).index[::-1].tolist())
e2fg2m = set(hall['HALLMARK_E2F_TARGETS']) | set(hall['HALLMARK_G2M_CHECKPOINT'])
print('Share of top 100 falling genes in E2F/G2M:', round(float(top.head(100).index.isin(e2fg2m).mean()), 3), '| rising:', round(float(top.tail(100).index.isin(e2fg2m).mean()), 3))
sym_to_ens = {}
gs = set(genes)
for e, sy in ens2sym_tbl.items():
    if sy in gs:
        sym_to_ens.setdefault(sy, []).append(e)
json.dump({'genes': genes, 'ensembl_by_symbol': sym_to_ens, 'beta': {'final': [float(f'{x:.4g}') for x in final_beta], **{f'without_{k}': [float(f'{x:.4g}') for x in betas[k]] for k in [ASC, BM] if k in betas}},
           'signature': {g: float(pub_sym[g]) for g in weights.index}, 'error': err.reset_index().astype({'dt_bin': str}).to_dict('records'), 'gate': gate}, open(RESULTS / 'virtual_passage_model.json', 'w'))
json.dump(gate, open(RESULTS / 'virtual_passage_gate.json', 'w'), indent=1)
(REPORTS / 'VIRTUAL_PASSAGE_RESULTS.md').write_text('# Virtual passage results\n\nGate: ' + json.dumps(gate, indent=1) + '\n\nBy cohort:\n' + by.round(3).to_string() + '\n\nBy distance:\n' + err.round(4).to_string() + '\n', encoding='utf-8')

# Descriptive only: does early movement relate to reaching growth arrest in the adipose cohort?
growth = {'A': 'arrested', 'B': 'arrested', 'C': 'arrested', 'D': 'stopped at p10', 'E': 'stopped at p10'}
g = [{'donor': s['subject'], 'status': growth.get(s['subject']), 'last_passage': s['passage'][-1], 'gamma_early': float(((s['X'][1] - s['X'][0]) @ betas[ASC]) / ((betas[ASC] @ betas[ASC]) * (tlog(s['passage'][1]) - tlog(s['passage'][0]))))} for s in PROFILES[ASC]]
print(pd.DataFrame(g).round(3).to_string(index=False))
pd.DataFrame(g).to_csv(RESULTS / 'virtual_passage_growth_descriptive.csv', index=False)